# Customer Churn Prediction

## Imports 

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load Data

In [2]:
df = pd.read_excel("../data/dataset.xlsx")

## EDA

In [3]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
df.tail()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
7038,2569-WGERO,1,United States,California,Landers,92285,"34.341737, -116.539416",34.341737,-116.539416,Female,...,Two year,Yes,Bank transfer (automatic),21.15,1419.4,No,0,45,5306,NaN
7039,6840-RESVB,1,United States,California,Adelanto,92301,"34.667815, -117.536183",34.667815,-117.536183,Male,...,One year,Yes,Mailed check,84.80,1990.5,No,0,59,2140,NaN
7040,2234-XADUH,1,United States,California,Amboy,92304,"34.559882, -115.637164",34.559882,-115.637164,Female,...,One year,Yes,Credit card (automatic),103.20,7362.9,No,0,71,5560,NaN
7041,4801-JZAZL,1,United States,California,Angelus Oaks,92305,"34.1678, -116.86433",34.167800,-116.864330,Female,...,Month-to-month,Yes,Electronic check,29.60,346.45,No,0,59,2793,NaN
7042,3186-AJIEK,1,United States,California,Apple Valley,92308,"34.424926, -117.184503",34.424926,-117.184503,Male,...,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No,0,38,5097,NaN


In [5]:
df.columns.to_list()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

### Removing identifier columns 

In [6]:
df = df.drop(columns= ['CustomerID','Count','Lat Long','Latitude','Longitude'])

### Removing columns that relate to the actual Churn Value. Including them would be data leakage

In [7]:
df = df.drop(columns = ["Churn Score", "Churn Label", "CLTV"])

In [8]:
df.columns.to_list()

['Country',
 'State',
 'City',
 'Zip Code',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Value',
 'Churn Reason']

### Checking missing values

In [9]:
df.isnull().sum()

Country                 0
State                   0
City                    0
Zip Code                0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Value             0
Churn Reason         5174
dtype: int64

#### Drop the Churn Reaso column. Because it has more than 50% missing values 

In [10]:
df = df.drop(columns = ["Churn Reason"])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Country            7043 non-null   object 
 1   State              7043 non-null   object 
 2   City               7043 non-null   object 
 3   Zip Code           7043 non-null   int64  
 4   Gender             7043 non-null   object 
 5   Senior Citizen     7043 non-null   object 
 6   Partner            7043 non-null   object 
 7   Dependents         7043 non-null   object 
 8   Tenure Months      7043 non-null   int64  
 9   Phone Service      7043 non-null   object 
 10  Multiple Lines     7043 non-null   object 
 11  Internet Service   7043 non-null   object 
 12  Online Security    7043 non-null   object 
 13  Online Backup      7043 non-null   object 
 14  Device Protection  7043 non-null   object 
 15  Tech Support       7043 non-null   object 
 16  Streaming TV       7043 

Total Charges  is identified as an object, but its unumeric. convert Total Charges data type to numeric

In [12]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors= "coerce")

when there are no charges it is 0, to prevent erros 

In [13]:
df["Total Charges"] = df["Total Charges"].fillna(0)

### Checking statistics of the columns 

In [14]:
df.describe()

,Zip Code,Tenure Months,Monthly Charges,Total Charges,Churn Value
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,93521.964646,32.371149,64.761692,2279.734304,0.265370
std,1865.794555,24.559481,30.090047,2266.794470,0.441561
min,90001.000000,0.000000,18.250000,0.000000,0.000000
25%,92102.000000,9.000000,35.500000,398.550000,0.000000
50%,93552.000000,29.000000,70.350000,1394.550000,0.000000
75%,95351.000000,55.000000,89.850000,3786.600000,1.000000
max,96161.000000,72.000000,118.750000,8684.800000,1.000000


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Country            7043 non-null   object 
 1   State              7043 non-null   object 
 2   City               7043 non-null   object 
 3   Zip Code           7043 non-null   int64  
 4   Gender             7043 non-null   object 
 5   Senior Citizen     7043 non-null   object 
 6   Partner            7043 non-null   object 
 7   Dependents         7043 non-null   object 
 8   Tenure Months      7043 non-null   int64  
 9   Phone Service      7043 non-null   object 
 10  Multiple Lines     7043 non-null   object 
 11  Internet Service   7043 non-null   object 
 12  Online Security    7043 non-null   object 
 13  Online Backup      7043 non-null   object 
 14  Device Protection  7043 non-null   object 
 15  Tech Support       7043 non-null   object 
 16  Streaming TV       7043 

## Preprocesing

#### Grouping numerical columns columns together, 
Preaparing them for scaling, so that they are within the same range,in case there  are outliers  

we dont include the "Churn Value" column in the list of numerical columns. because that is what we are going to be predicting 

In [16]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop("Churn Value")

In [17]:
num_cols

Index(['Zip Code', 'Tenure Months', 'Monthly Charges', 'Total Charges'], dtype='object')

#### Grouping categorical columns together

In [18]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns

fill missing strings with the word "missing", so they dont couse an error 

In [19]:
df[cat_cols] = df[cat_cols].fillna("Missing").astype(str)

In [20]:
cat_cols

Index(['Country', 'State', 'City', 'Gender', 'Senior Citizen', 'Partner',
       'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service',
       'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
       'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
       'Payment Method'],
      dtype='object')

#### Create X by combining the categorical columns and the numerical columns 

In [21]:
X = pd.concat([df[num_cols], df[cat_cols]], axis=1)
y = df["Churn Value"]

#### Split the data before scaling or encoding, to prevent data leakage

In [22]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size = 0.2, random_state = 42)

#### Scale all numerical columns for the test and training set

In [23]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# dont use fit on the testing set, prvent data leakage
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [24]:
X_test[cat_cols]

,Country,State,City,Gender,Senior Citizen,Partner,Dependents,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method
185,United States,California,Keene,Female,No,No,No,Yes,No,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic)
2715,United States,California,Stockton,Male,No,Yes,No,Yes,Yes,DSL,No,Yes,No,No,No,No,Month-to-month,No,Credit card (automatic)
3825,United States,California,San Leandro,Female,No,No,No,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check
1807,United States,California,Sherman Oaks,Male,Yes,Yes,No,Yes,No,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,Yes,Electronic check
132,United States,California,Murrieta,Male,No,No,No,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,No,Bank transfer (automatic)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6366,United States,California,Copperopolis,Female,Yes,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Electronic check
315,United States,California,Modesto,Male,No,No,No,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic)
2439,United States,California,Inyokern,Female,No,Yes,No,Yes,No,Fiber optic,Yes,Yes,No,No,Yes,No,Month-to-month,No,Electronic check
5002,United States,California,Danville,Male,No,Yes,Yes,Yes,Yes,Fiber optic,Yes,Yes,Yes,No,Yes,Yes,Two year,Yes,Electronic check


In [25]:

X_train[cat_cols]

,Country,State,City,Gender,Senior Citizen,Partner,Dependents,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method
2142,United States,California,San Diego,Male,No,Yes,No,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic)
1623,United States,California,Jamestown,Female,No,No,No,Yes,No,Fiber optic,No,Yes,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic)
6074,United States,California,Littlerock,Female,No,No,No,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic)
1362,United States,California,El Monte,Female,Yes,Yes,No,No,No phone service,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic)
6754,United States,California,Los Angeles,Female,No,Yes,Yes,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,United States,California,San Francisco,Female,No,Yes,Yes,Yes,No,Fiber optic,No,No,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check
5191,United States,California,Keyes,Female,No,No,No,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic)
5226,United States,California,Clearlake,Male,No,Yes,Yes,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check
5390,United States,California,Sacramento,Male,No,Yes,Yes,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Electronic check


#### encode the categorical columns using one hot encoding because they are categorical columns 

In [26]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_cat_encoded = encoder.fit_transform(X_train[cat_cols])

X_test_cat_encoded = encoder.transform(X_test[cat_cols])




In [27]:
X_train_cat = pd.DataFrame(X_train_cat_encoded, columns = encoder.get_feature_names_out(cat_cols), index = X_train.index)
X_test_cat = pd.DataFrame(X_test_cat_encoded, columns = encoder.get_feature_names_out(cat_cols), index = X_test.index)

In [28]:
X_train= pd.concat([X_train[num_cols] , X_train_cat ], axis=1)
X_test = pd.concat([X_test[num_cols] , X_test_cat ], axis=1)

#### Create the model called "myModel" and train it

In [29]:
# creating the model
myModel = LogisticRegression()  


# trainging the model
myModel.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Model Evaluation 

In [30]:
# Testing model by preding y
y_pred = myModel.predict(X_test)

In [31]:
print(f"Logistic Regression model accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Logistic Regression model accuracy: 79.49%


In [32]:
print(confusion_matrix(y_test, y_pred ))
print(classification_report(y_test, y_pred))

[[892 117]
 [172 228]]
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1009
           1       0.66      0.57      0.61       400

    accuracy                           0.79      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.79      0.79      0.79      1409

